<a href="https://colab.research.google.com/github/Kornieks/NYC-Job-Postings/blob/main/NY_Job_postings_Data_Prep.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from scipy import stats
import numpy as np
import re

# For text analysis
from sklearn.feature_extraction.text import CountVectorizer

# Initial Data Inspection

In [ ]:
# Load Raw Data

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%cd /content/drive/MyDrive/Datasets

/content/drive/MyDrive/Datasets


In [ ]:
jobs_df = pd.read_csv('Jobs_NYC_Postings.csv')
jobs_df.head()

,Job ID,Agency,Posting Type,# Of Positions,Business Title,Civil Service Title,Title Classification,Title Code No,Level,Job Category,...,Additional Information,To Apply,Hours/Shift,Work Location 1,Recruitment Contact,Residency Requirement,Posting Date,Post Until,Posting Updated,Process Date
0,767684,DEPT OF ENVIRONMENT PROTECTION,Internal,1,"SECTION MANAGER, QUALITY ASSURANCE",ADMINISTRATIVE ENGINEER,Competitive-1,10015,M3,"Administration & Human Resources Engineering, ...",...,NaN,NaN,NaN,96-05 Horace Harding Expway,NaN,New York City Residency is not required for th...,02/02/2026,03-APR-2026,02/02/2026,03/17/2026
1,774000,DEPARTMENT OF BUILDINGS,Internal,1,Plan Examination Intern (Manhattan),SUMMER COLLEGE INTERN,Non-Competitive-5,10234,00,"Engineering, Architecture, & Planning",...,NaN,NaN,NaN,"280 Broadway, 3rd Floor, N.Y.",NaN,This internship does not require residency in ...,03/13/2026,27-MAR-2026,03/16/2026,03/17/2026
2,765029,NYC HOUSING AUTHORITY,Internal,1,Housing Assistant,HOUSING ASSISTANT (HA),Competitive-1,80201,00,Administration & Human Resources,...,1.\tFor NYCHA employees: this position is open...,NaN,NaN,LHD-Client Services Admin,NaN,NYCHA has no residency requirements.,02/27/2026,19-MAR-2026,02/27/2026,03/17/2026
3,770645,OFFICE OF THE MAYOR,External,1,Deputy Borough Director - Bronx,Special Assistant (MA)-MGRL,Pending Classification-2,0668A,M6,Constituent Services & Community Programs,...,NaN,NaN,NaN,253 Broadway New York Ny,NaN,New York City residency is generally required ...,02/17/2026,18-APR-2026,02/17/2026,03/17/2026
4,773101,HOUSING PRESERVATION & DVLPMNT,Internal,2,Agency Attorney L2 for the Housing Litigation ...,AGENCY ATTORNEY,Non-Competitive-5,30087,02,Legal Affairs,...,NaN,NaN,NaN,100 Gold Street,NaN,New York City Residency is not required for th...,03/09/2026,08-APR-2026,03/09/2026,03/17/2026


In [ ]:
# Number of rows and columns

jobs_df.shape

(2692, 30)

In [ ]:
# General info about the dataset

jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2692 entries, 0 to 2691
Data columns (total 30 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Job ID                         2692 non-null   int64  
 1   Agency                         2692 non-null   object 
 2   Posting Type                   2692 non-null   object 
 3   # Of Positions                 2692 non-null   int64  
 4   Business Title                 2692 non-null   object 
 5   Civil Service Title            2692 non-null   object 
 6   Title Classification           2692 non-null   object 
 7   Title Code No                  2692 non-null   object 
 8   Level                          2692 non-null   object 
 9   Job Category                   2692 non-null   object 
 10  Full-Time/Part-Time indicator  2692 non-null   object 
 11  Career Level                   2692 non-null   object 
 12  Salary Range From              2692 non-null   f

## Overview
**Rows and columns:**


*   The dataset contains 2692 rows and 30 columns.
*   It contains current job postings available on the City of [New York’s official jobs site](http://www.nyc.gov/html/careers/html/search/search.shtml).
* Internal postings available to city employees and external postings available to the general public are included.
* More information is available through this [link](https://github.com/Kornieks/NYC-Job-Postings/blob/main/Dataset%20description.md)

The following columns contain **missing values:**
*   Minimum Qual Requirements
*   Preferred skills
*   Additional Information
*   To apply
*   Hours/Shift
*   Recruitment Contact
*   Post Until

The following columns have **incorrect data types:**
*   Posting Date, Post Until, Posting Updated, Process Date should be datatime;



# Data Cleaning

In [ ]:
# Changing datatype to date

jobs_df['Post Until'] = pd.to_datetime(jobs_df['Post Until'])
jobs_df['Posting Date'] = pd.to_datetime(jobs_df['Posting Date'])
jobs_df['Posting Updated'] = pd.to_datetime(jobs_df['Posting Updated'])

/tmp/ipykernel_8027/3225058878.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  jobs_df['Post Until'] = pd.to_datetime(jobs_df['Post Until'])


In [ ]:
print(f'Median Posting Duration: {(jobs_df['Post Until'] - jobs_df['Posting Date']).median()}')
print(f'\nAverage posting duration: {(jobs_df['Post Until'] - jobs_df['Posting Date']).mean()}')

Median Posting Duration: 60 days 00:00:00

Average posting duration: 53 days 15:09:31.812080537


In [ ]:
jobs_df['Posting Type'].value_counts()

,count
Posting Type,
External,1405
Internal,1287


In [ ]:
jobs_df = jobs_df[jobs_df['Posting Type'] == 'External']
jobs_df = jobs_df.drop('Posting Type', axis=1)

Internal postings were filtered out because the goal of this analysis is to understand the public job market rather than internal hiring opportunities. Internal postings are intended for current employees only, so including them could distort the results by mixing two different hiring audiences.
Therefore, the column "Posting type" was dropped.


### Work with missing values

In [ ]:
# Calculating numbers of missing values per column

jobs_df.isna().sum()

,0
Job ID,0
Agency,0
# Of Positions,0
Business Title,0
Civil Service Title,0
Title Classification,0
Title Code No,0
Level,0
Job Category,0
Full-Time/Part-Time indicator,0


In [ ]:
# Calculating percentage of missing values per column

jobs_df.isna().sum() * 100 / jobs_df.shape[0]

,0
Job ID,0.000000
Agency,0.000000
# Of Positions,0.000000
Business Title,0.000000
Civil Service Title,0.000000
Title Classification,0.000000
Title Code No,0.000000
Level,0.000000
Job Category,0.000000
Full-Time/Part-Time indicator,0.000000


In [ ]:
# Replacing none values with 'Not Specified'

jobs_df['Minimum Qual Requirements'] = jobs_df['Minimum Qual Requirements'].fillna('Not Specified')
jobs_df['Preferred Skills'] = jobs_df['Preferred Skills'].fillna('Not Specified')

In [ ]:
# Deleting irrelevant columns

jobs_df = jobs_df.drop(['Title Classification', 'Title Code No', 'Level', 'Work Location', 'Work Location 1', 'Division/Work Unit', 'Job Description', 'Additional Information', 'Hours/Shift', 'To Apply', 'Process Date', 'Post Until', 'Recruitment Contact', ], axis=1)

The “Recruitment Contact” column was deleted because it contains no data.
The columns "Additional Information," "Job Description," "Level," and "To Apply", "Process Date", "Title Code No", "Division/Work Unit", "Work Location 1", "Post Until" were dropped as they were not relevant to the scope of this analysis.

In [ ]:
# Check

jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1405 entries, 3 to 2688
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   Job ID                         1405 non-null   int64         
 1   Agency                         1405 non-null   object        
 2   # Of Positions                 1405 non-null   int64         
 3   Business Title                 1405 non-null   object        
 4   Civil Service Title            1405 non-null   object        
 5   Job Category                   1405 non-null   object        
 6   Full-Time/Part-Time indicator  1405 non-null   object        
 7   Career Level                   1405 non-null   object        
 8   Salary Range From              1405 non-null   float64       
 9   Salary Range To                1405 non-null   float64       
 10  Salary Frequency               1405 non-null   object        
 11  Minimum Qual Requireme

### Work with duplicates

In [ ]:
# Check whether there are duplicated rows

jobs_df.duplicated().sum()

np.int64(2)

In [ ]:
# Dropping duplicated rows

jobs_df = jobs_df.drop_duplicates()

In [ ]:
# Check whether there are duplicated Job postings

jobs_df['Job ID'].value_counts().sort_values(ascending=False)

,count
Job ID,
721324,2
765171,2
712968,2
771447,1
764940,1
...,...
773474,1
764073,1
769726,1


In [ ]:
jobs_df[jobs_df['Job ID'] == 765171].T

,268,448
Job ID,765171,765171
Agency,DEPT OF HEALTH/MENTAL HYGIENE,DEPT OF HEALTH/MENTAL HYGIENE
# Of Positions,75,75
Business Title,"Public Health Nurse Part-Time), Bureau of Scho...","Public Health Nurse Part-Time), Bureau of Scho..."
Civil Service Title,PUBLIC HEALTH NURSE (SCHOOL HE,PUBLIC HEALTH NURSE (SCHOOL HE
Job Category,Health,Health
Full-Time/Part-Time indicator,P,P
Career Level,Experienced (non-manager),Experienced (non-manager)
Salary Range From,48.83,48.83
Salary Range To,48.83,48.83


In [ ]:
# Deleting Job ID duplicates

jobs_df = jobs_df.sort_values(by='Posting Date').drop_duplicates(subset='Job ID', keep = 'last')

In [ ]:
# Unique values in columns

jobs_df.nunique()

,0
Job ID,1400
Agency,56
# Of Positions,27
Business Title,1095
Civil Service Title,295
Job Category,98
Full-Time/Part-Time indicator,2
Career Level,5
Salary Range From,357
Salary Range To,459


In [ ]:
# Renaming the Part/Full time column

jobs_df = jobs_df.rename(columns={'Full-Time/Part-Time indicator': 'Work Schedule Type'})

In [ ]:
print(
      f"Work schedule Type: {jobs_df['Work Schedule Type'].unique()}",
      f"\nCareer Levels: {jobs_df['Career Level'].unique()}",
      f"\nSalary Frequency: {jobs_df['Salary Frequency'].unique()}")

Work schedule Type: ['F' 'P'] 
Career Levels: ['Experienced (non-manager)' 'Entry-Level' 'Manager' 'Student' 'Executive'] 
Salary Frequency: ['Annual' 'Hourly' 'Daily']


Some postings appeared multiple times because the same vacancy was published more than once. For analyses focused on unique vacancies, these records were deduplicated by Job ID, keeping last posted.

### Look for extreme values

In [ ]:

jobs_df['# Of Positions'].describe()

,# Of Positions
count,1400.000000
mean,2.116429
std,8.814992
min,1.000000
25%,1.000000
50%,1.000000
75%,1.000000
max,243.000000


In [ ]:
jobs_df['# Of Positions'].value_counts()

,count
# Of Positions,
1,1172
2,101
3,39
4,22
10,11
5,11
6,7
8,6
20,4


In [ ]:
# Inspecting high values

jobs_df[jobs_df['# Of Positions'] > 20][['Job ID', 'Agency', 'Business Title', '# Of Positions', 'Posting Date']].sort_values(by='# Of Positions', ascending=False)

,Job ID,Agency,Business Title,# Of Positions,Posting Date
219,769895,DEPT OF PARKS & RECREATION,City Seasonal Aide/Security Guard,243,2026-02-12
2334,733770,ADMIN FOR CHILDREN'S SVCS,Child Protective Specialist,130,2026-03-05
1456,765438,POLICE DEPARTMENT,Assistant School Safety Agent,100,2026-01-30
448,765171,DEPT OF HEALTH/MENTAL HYGIENE,"Public Health Nurse Part-Time), Bureau of Scho...",75,2026-03-09
1351,769724,DEPT OF YOUTH & COMM DEV SRVS,SYEP Community Program Specialists-Senior Fiel...,70,2026-02-10
1573,769071,LAW DEPARTMENT,SUMMER GRADUATE INTERN,50,2026-02-25
2257,760053,ADMIN FOR CHILDREN'S SVCS,Youth Development Specialist,50,2026-03-05
2329,771279,DEPARTMENT OF BUILDINGS,Construction Inspector,47,2026-02-25
1279,773702,DEPT OF PARKS & RECREATION,Aquatics Specialist,40,2026-03-13
2630,771710,LAW DEPARTMENT,College Aide Level 1,27,2026-02-26


The # Of Positions field represents the number of vacancies attached to a posting. While most postings hire for 1 role, some include bulk hiring and therefore show large values. Extreme counts were reviewed separately as potential outliers.

In [ ]:
# Splitting the data by salary frequency

annual_df = jobs_df[jobs_df['Salary Frequency'] == 'Annual'].copy()
daily_df = jobs_df[jobs_df['Salary Frequency'] == 'Daily'].copy()
hourly_df = jobs_df[jobs_df['Salary Frequency'] == 'Hourly'].copy()

In [ ]:
annual_df[['Salary Range From', 'Salary Range To']].describe()

,Salary Range From,Salary Range To
count,1136.000000,1136.000000
mean,77903.786734,101684.257861
std,31762.834105,43161.778774
min,0.000000,28.980000
25%,59888.000000,70904.000000
50%,68443.000000,93577.000000
75%,87743.000000,120000.000000
max,278769.000000,285413.000000


In [ ]:
# Outlier Detection - InterQuartile Range

q1 = annual_df['Salary Range From'].quantile(0.25)
q3 = annual_df['Salary Range From'].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - iqr * 1.5
upper_bound = q3 + iqr * 1.5

annual_df[(annual_df['Salary Range From'] < lower_bound) | (annual_df['Salary Range From'] > upper_bound)]

,Job ID,Agency,# Of Positions,Business Title,Civil Service Title,Job Category,Work Schedule Type,Career Level,Salary Range From,Salary Range To,Salary Frequency,Minimum Qual Requirements,Preferred Skills,Residency Requirement,Posting Date,Posting Updated
1673,766593,NYC EMPLOYEES RETIREMENT SYS,1,Data Acquisition Manager,COMPUTER SYSTEMS MANAGER,"Technology, Data & Innovation",F,Manager,150000.0,160000.0,Annual,1. A master's degree in computer science from ...,"• Hands-on knowledge of MS SQL Server, DB2 LUW...",New York City Residency is not required for th...,2026-01-18,2026-01-16
2549,767496,OFFICE OF LABOR RELATIONS,1,Director of Communications,EXECUTIVE PROGRAM SPECIALIST,Communications & Intergovernmental Affairs Pol...,F,Manager,150000.0,180000.0,Annual,Qualifications Requirements 1.\tA bachelor’s d...,Not Specified,New York City residency is generally required ...,2026-01-30,2026-02-19
1172,769357,BRONX DISTRICT ATTORNEY,1,Director of Application Development,COMPUTER SYSTEMS MANAGER,"Technology, Data & Innovation",F,Manager,180791.0,180791.0,Annual,1. A master's degree in computer science from ...,- Proven experience modernizing legacy systems...,City Residency is not required for this position.,2026-02-06,2026-02-06
719,769189,TAXI & LIMOUSINE COMMISSION,1,Chief of Staff,CHIEF OF STAFF (TLC),Administration & Human Resources,F,Executive,150000.0,195000.0,Annual,Not Specified,The preferred candidate should possess the fol...,New York City residency is generally required ...,2026-02-06,2026-02-05
1328,769321,HOUSING PRESERVATION & DVLPMNT,1,Deputy General Counsel for the Landlord Tenant...,EXECUTIVE AGENCY COUNSEL,Legal Affairs,F,Manager,170000.0,180000.0,Annual,Admission to the New York State Bar; and four ...,Not Specified,New York City Residency is not required for th...,2026-02-06,2026-02-06
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2211,774145,OFFICE OF THE MAYOR,1,"Program Manager, Infrastructure",RESEARCH PROJECTS COOR(MA)-MGR,"Engineering, Architecture, & Planning Policy, ...",F,Experienced (non-manager),145000.0,165000.0,Annual,1. Do you have a baccalaureate degree from an ...,•\tYou can support the creation of new policie...,New York City residency is generally required ...,2026-03-16,2026-03-16
1560,774152,OFFICE OF THE MAYOR,1,Policy Advisor for Public Safety,RESEARCH PROJECTS COOR(MA)-MGR,"Policy, Research & Analysis",F,Manager,140000.0,175000.0,Annual,1. Do you have a baccalaureate degree from an ...,•\tYou are familiar with the NYC public safety...,New York City residency is generally required ...,2026-03-16,2026-03-16
2418,774201,HOUSING PRESERVATION & DVLPMNT,1,Deputy Commissioner for External Affairs / Age...,DEPUTY COMMISSIONER (HOUSING P,Communications & Intergovernmental Affairs,F,Executive,207000.0,207000.0,Annual,1. A baccalaureate degree from an accredited c...,Not Specified,New York City residency is generally required ...,2026-03-16,2026-03-16
1200,752860,FIRE DEPARTMENT,1,FIRE MEDICAL OFFICER,FIRE MEDICAL OFFICER,Health,F,Experienced (non-manager),154636.0,182278.0,Annual,Possession of a valid license to practice medi...,This position may require evenings and weekend...,New York City residency is generally required ...,2026-03-16,2026-03-16


In [ ]:
# Outlier Detection - Z-score

annual_z_scores = np.abs(stats.zscore(annual_df['Salary Range From'].dropna()))

annual_df[annual_z_scores > 3]  # >3 standard deviations

,Job ID,Agency,# Of Positions,Business Title,Civil Service Title,Job Category,Work Schedule Type,Career Level,Salary Range From,Salary Range To,Salary Frequency,Minimum Qual Requirements,Preferred Skills,Residency Requirement,Posting Date,Posting Updated
1172,769357,BRONX DISTRICT ATTORNEY,1,Director of Application Development,COMPUTER SYSTEMS MANAGER,"Technology, Data & Innovation",F,Manager,180791.00,180791.00,Annual,1. A master's degree in computer science from ...,- Proven experience modernizing legacy systems...,City Residency is not required for this position.,2026-02-06,2026-02-06
2598,769843,DEPT OF HEALTH/MENTAL HYGIENE,1,"BMIRH Medical Director, Bureau of Maternal Inf...",CITY MEDICAL SPECIALIST,Health,F,Experienced (non-manager),187816.00,201267.00,Annual,Possession of a valid license to practice medi...,Not Specified,New York City residency is generally required ...,2026-02-18,2026-02-18
134,771550,OFFICE OF THE MAYOR,1,Senior Advisor – Housing & Homelessness,RESEARCH PROJECTS COOR(MA)-MGR,"Policy, Research & Analysis",F,Experienced (non-manager),180000.00,190000.00,Annual,1. Do you have a baccalaureate degree from an ...,•\tDemonstrated experience in task and informa...,New York City residency is generally required ...,2026-02-25,2026-02-25
2071,771472,DEPT OF ENVIRONMENT PROTECTION,1,"Deputy Commissioner, Bureau of Water Supply",ADMINISTRATIVE PROJECT MANAGER,"Engineering, Architecture, & Planning",F,Executive,278769.00,278769.00,Annual,1. A baccalaureate degree from an accredited c...,Not Specified,New York City residency is not required for th...,2026-02-27,2026-02-27
1273,771744,DEPT OF PARKS & RECREATION,1,Borough Commissioner,BOROUGH COMMISSIONER (PARKS AN,Administration & Human Resources Constituent S...,F,Manager,190000.00,205000.00,Annual,Qualifications are determined by Agency Head.,1.\tFive plus years of management experience a...,Residency in one of the five boroughs of New Y...,2026-02-27,2026-02-26
1797,771584,DEPT OF ENVIRONMENT PROTECTION,1,"Deputy Commissioner, Bureau of Water Supply",ADMINISTRATIVE ENGINEER,"Engineering, Architecture, & Planning",F,Executive,278769.00,278769.00,Annual,1. A valid New York State License as a Profess...,Not Specified,New York City residency is not required for th...,2026-02-27,2026-02-27
1818,768358,DEPARTMENT OF FINANCE,1,"Assistant Commissioner, Office of Employee Ser...",ADMINISTRATIVE STAFF ANALYST,Administration & Human Resources Communication...,F,Manager,200000.00,215000.00,Annual,1. A master's degree from an accredited colleg...,•\tCandidate must have at least 10 years full-...,New York City residency is generally required ...,2026-02-27,2026-02-27
1773,772077,OFFICE OF THE MAYOR,1,Senior Advisor,Special Assistant (MA)-MGRL,Communications & Intergovernmental Affairs,F,Manager,175000.00,185000.00,Annual,•\tBachelor’s Degree required. •\t7+ years of...,Not Specified,New York City residency is generally required ...,2026-03-02,2026-03-02
611,772008,DEPT OF ENVIRONMENT PROTECTION,1,"Director, Personnel Services",ADMINISTRATIVE STAFF ANALYST,Administration & Human Resources Communication...,F,Manager,195000.00,195000.00,Annual,1. A master's degree from an accredited colleg...,Not Specified,New York City residency is not required for th...,2026-03-02,2026-03-04
1449,772246,OFFICE OF THE MAYOR,1,Criminal Justice Coordinator,RESEARCH PROJECTS COOR(MA)-MGR,"Policy, Research & Analysis",F,Manager,180000.00,230000.00,Annual,1. Do you have a baccalaureate degree from an ...,•\tYou are highly familiar with NYC’s criminal...,New York City residency is generally required ...,2026-03-03,2026-03-03


In [ ]:
daily_df[['Salary Range From', 'Salary Range To']].describe()

,Salary Range From,Salary Range To
count,8.00000,8.000000
mean,393.90875,422.628750
std,82.41813,64.750232
min,320.80000,331.920000
25%,342.90000,397.560000
50%,346.56000,403.440000
75%,446.65250,446.652500
max,518.00000,518.000000


In [ ]:
# Outlier Detection - Z-score

daily_z_scores = np.abs(stats.zscore(daily_df['Salary Range From'].dropna()))

daily_df[daily_z_scores > 3]  # >3 standard deviations

,Job ID,Agency,# Of Positions,Business Title,Civil Service Title,Job Category,Work Schedule Type,Career Level,Salary Range From,Salary Range To,Salary Frequency,Minimum Qual Requirements,Preferred Skills,Residency Requirement,Posting Date,Posting Updated


In [ ]:
hourly_df[['Salary Range From', 'Salary Range To']].describe()

,Salary Range From,Salary Range To
count,256.000000,256.000000
mean,23.107624,25.363560
std,16.374138,17.540473
min,16.500000,17.000000
25%,17.000000,17.000000
50%,17.500000,19.500000
75%,18.082500,25.170000
max,100.000000,118.000000


In [ ]:
# Outlier Detection - Z-score

hourly_z_scores = np.abs(stats.zscore(hourly_df['Salary Range From'].dropna()))

hourly_df[hourly_z_scores > 3] # >3 standard deviations

,Job ID,Agency,# Of Positions,Business Title,Civil Service Title,Job Category,Work Schedule Type,Career Level,Salary Range From,Salary Range To,Salary Frequency,Minimum Qual Requirements,Preferred Skills,Residency Requirement,Posting Date,Posting Updated
2484,768257,DEPT OF HEALTH/MENTAL HYGIENE,1,"Physician, Bureau of Tuberculosis Control",CITY MEDICAL SPECIALIST (PART-,Health,F,Experienced (non-manager),92.08,98.61,Hourly,Possession of a valid license to practice medi...,Not Specified,New York City Residency is not required for th...,2026-01-29,2026-01-29
450,757711,DEPT OF HEALTH/MENTAL HYGIENE,1,"Supervising Physician K-8, Bureau of School He...",CITY MEDICAL SPECIALIST (PART-,Health,P,Experienced (non-manager),92.08,98.61,Hourly,Possession of a valid license to practice medi...,Not Specified,New York City Residency is not required for th...,2026-02-17,2026-02-17
2370,699810,DEPT OF HEALTH/MENTAL HYGIENE,1,"Adolescent Medicine – Physician, Bureau of Sch...",CITY MEDICAL SPECIALIST (PART-,Health,P,Experienced (non-manager),79.23,79.23,Hourly,Possession of a valid license to practice medi...,Not Specified,New York City Residency is not required for th...,2026-02-18,2026-02-18
303,760653,DEPT OF HEALTH/MENTAL HYGIENE,1,"Pediatric School Health Field Physician, Burea...",CITY MEDICAL SPECIALIST (PART-,Health,F,Experienced (non-manager),92.08,98.61,Hourly,Possession of a valid license to practice medi...,Not Specified,New York City Residency is not required for th...,2026-02-18,2026-02-18
821,750297,DEPT OF HEALTH/MENTAL HYGIENE,1,"Assistant Medical Director of High Schools, Bu...",CITY MEDICAL SPECIALIST (PART-,Health,P,Experienced (non-manager),92.08,98.61,Hourly,Possession of a valid license to practice medi...,Not Specified,New York City Residency is not required for th...,2026-02-18,2026-02-18
392,760773,DEPT OF HEALTH/MENTAL HYGIENE,1,"Pediatrics School Health Field Physician, Burt...",CITY MEDICAL SPECIALIST (PART-,Health,P,Experienced (non-manager),92.08,98.61,Hourly,Possession of a valid license to practice medi...,Not Specified,New York City Residency is not required for th...,2026-02-18,2026-02-18
2268,698355,DEPT OF HEALTH/MENTAL HYGIENE,1,"School Health Field Physician, Bureau of Schoo...",CITY MEDICAL SPECIALIST (PART-,Health,P,Experienced (non-manager),79.23,79.23,Hourly,Possession of a valid license to practice medi...,Not Specified,New York City Residency is not required for th...,2026-02-18,2026-02-18
2275,757521,DEPT OF HEALTH/MENTAL HYGIENE,1,"Pediatric Pulmonologist, Bureau of School Health",CITY MEDICAL SPECIALIST (PART-,Health,P,Experienced (non-manager),92.08,98.61,Hourly,Possession of a valid license to practice medi...,Not Specified,New York City Residency is not required for th...,2026-02-18,2026-02-18
984,698248,DEPT OF HEALTH/MENTAL HYGIENE,1,"School Health Field Physician, Bureau of Schoo...",CITY MEDICAL SPECIALIST (PART-,Health,P,Experienced (non-manager),79.23,79.23,Hourly,Possession of a valid license to practice medi...,Not Specified,New York City Residency is not required for th...,2026-02-18,2026-02-18
1764,686469,DEPT OF HEALTH/MENTAL HYGIENE,1,"Supervising Medical Doctor, Bureau of School H...",CITY MEDICAL SPECIALIST (PART-,Health,P,Experienced (non-manager),79.23,84.86,Hourly,Possession of a valid license to practice medi...,Not Specified,New York City Residency is not required for th...,2026-02-18,2026-02-18


In [ ]:
jobs_df.shape

(1400, 16)

In [ ]:
jobs_df['Posting Date'].min()

Timestamp('2025-03-25 00:00:00')

In [ ]:
# Deleting one row which contains a position posted last year (outlier)

jobs_df = jobs_df.drop(jobs_df[jobs_df['Posting Date'] == '2025-03-25 00:00:00'].index)

In [ ]:
jobs_df['Posting Date'].min()

Timestamp('2026-01-16 00:00:00')

In [ ]:
jobs_df['Posting Date'].max()

Timestamp('2026-03-17 00:00:00')

## Conclusions:

**Datatypes:**
The columns `Post Until`, `Posting Date`, and `Posting Updated` were converted to datetime format to enable time-based analysis.

**Filtering:**
Internal postings were filtered out because the goal of this analysis is to understand the public job market. Internal postings are intended for current employees only, so including them could distort the results by mixing two different hiring audiences. Therefore, the column `Posting Type` was dropped.

**Missing values:**
Missing values in `Minimum Qual Requirements` and `Preferred Skills` were filled with `'Not Specified'` label as they were not critical for the analysis.

**Irrelevant columns:**
The following columns were dropped as they were either administrative, redundant, or not relevant to the scope of this analysis: `Title Classification`, `Title Code No`, `Level`, `Work Location`, `Work Location 1`, `Division/Work Unit`, `Job Description`, `Additional Information`, `Hours/Shift`, `To Apply`, `Process Date`, `Post Until`, `Recruitment Contact`.

**Duplicates:**
2 fully duplicated rows were found and removed. Additionally, the dataset was checked for duplicate `Job ID` values — some postings appeared more than once because the same vacancy was published multiple times. These were deduplicated by `Job ID`, keeping the most recently posted record.

**Anomalies:**
Outlier detection was performed separately for each salary frequency group: Annual, Daily, and Hourly. Both IQR and Z-score methods were applied. No extreme outliers were identified across the groups.

The `# Of Positions` field was reviewed separately. While most postings hire for 1 role, some postings showed unusually high values, representing bulk hiring. These were flagged and reviewed but retained in the dataset as they represent legitimate hiring activity.

In [ ]:
print(f'Jobs Posted Between: {jobs_df['Posting Date'].max()-jobs_df['Posting Date'].min()}')

Jobs Posted Between: 60 days 00:00:00


In [ ]:
jobs_df['Period before Updated'] = (jobs_df['Posting Updated']-jobs_df['Posting Date'])
jobs_df['Period before Updated']

,Period before Updated
873,47 days
2011,3 days
647,0 days
989,0 days
1110,3 days
...,...
1830,-1 days
578,-1 days
903,-1 days
1463,-1 days


**Posting Period:**
The difference between `Posting Date` and `Posting Updated` was calculated.
Most postings show positive values, meaning they were updated after the initial posting date.
However, some records show negative values (-1 days), indicating that `Posting Updated`
is earlier than `Posting Date`. This may be caused by data entry inconsistencies
or timezone differences and should be interpreted with caution.

## Categorization



New categorical columns need to be created to group high-cardinality fields into meaningful segments.
This i going to reduce noise, improve readability, and make the data more suitable for analysis and visualization.

In [ ]:
jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1399 entries, 873 to 224
Data columns (total 17 columns):
 #   Column                     Non-Null Count  Dtype          
---  ------                     --------------  -----          
 0   Job ID                     1399 non-null   int64          
 1   Agency                     1399 non-null   object         
 2   # Of Positions             1399 non-null   int64          
 3   Business Title             1399 non-null   object         
 4   Civil Service Title        1399 non-null   object         
 5   Job Category               1399 non-null   object         
 6   Work Schedule Type         1399 non-null   object         
 7   Career Level               1399 non-null   object         
 8   Salary Range From          1399 non-null   float64        
 9   Salary Range To            1399 non-null   float64        
 10  Salary Frequency           1399 non-null   object         
 11  Minimum Qual Requirements  1399 non-null   object         
 

In [ ]:
print(f'Number of unique job categories: {jobs_df['Job Category'].nunique()}')
jobs_df['Job Category'].unique()

Number of unique job categories: 98


array(['Engineering, Architecture, & Planning',
       'Technology, Data & Innovation Policy, Research & Analysis',
       'Administration & Human Resources',
       'Technology, Data & Innovation', 'Legal Affairs',
       'Constituent Services & Community Programs Social Services',
       'Engineering, Architecture, & Planning Public Safety, Inspections, & Enforcement',
       'Finance, Accounting, & Procurement',
       'Health Public Safety, Inspections, & Enforcement', 'Health',
       'Communications & Intergovernmental Affairs',
       'Public Safety, Inspections, & Enforcement', 'Social Services',
       'Constituent Services & Community Programs',
       'Policy, Research & Analysis Public Safety, Inspections, & Enforcement',
       'Administration & Human Resources Health',
       'Health Social Services',
       'Constituent Services & Community Programs Legal Affairs',
       'Administration & Human Resources Engineering, Architecture, & Planning',
       'Finance, Accountin

In [ ]:

canonical_categories = [
    'Administration & Human Resources',
    'Building Operations & Maintenance',
    'Communications & Intergovernmental Affairs',
    'Constituent Services & Community Programs',
    'Engineering, Architecture, & Planning',
    'Finance, Accounting, & Procurement',
    'Green Jobs',
    'Health',
    'Legal Affairs',
    'Mental Health',
    'Policy, Research & Analysis',
    'Public Safety, Inspections, & Enforcement',
    'Social Services',
    'Technology, Data & Innovation'
]

# Function to extract all matching categories from a raw text value
def extract_categories (text):
  text = str(text)
  found = [cat for cat in canonical_categories if cat.lower() in text.lower()]
  return found if found else ['Other']

jobs_df['Job Category Parsed'] = jobs_df['Job Category'].apply(extract_categories)

jobs_df['Category Count'] = jobs_df['Job Category Parsed'].apply(len)

jobs_df['Primary Job Category'] = jobs_df['Job Category Parsed'].apply(lambda x: x[0])

print(f'Number of categories in one job posting: {jobs_df['Category Count'].value_counts()}')

print(jobs_df['Primary Job Category'].value_counts())


Number of categories in one job posting: Category Count
1    1084
2     262
3      43
4       7
5       3
Name: count, dtype: int64
Primary Job Category
Engineering, Architecture, & Planning         291
Administration & Human Resources              146
Constituent Services & Community Programs     138
Health                                        137
Legal Affairs                                 132
Finance, Accounting, & Procurement            100
Building Operations & Maintenance              98
Technology, Data & Innovation                  79
Policy, Research & Analysis                    75
Social Services                                68
Public Safety, Inspections, & Enforcement      68
Communications & Intergovernmental Affairs     63
Green Jobs                                      4
Name: count, dtype: int64


The `Job Category` column originally contained 98 unique values, many of which were combinations
of multiple categories in a single string. To address this, a canonical list of categories was
defined and each posting was parsed to extract all matching categories.

A new column `Primary Job Category` was created by taking the first matched category per posting,
which is used as the main category label for analysis and visualization.

In [ ]:
print(f'Number of unique agencies: {jobs_df['Agency'].nunique()}')
jobs_df['Agency'].unique()

Number of unique agencies: 56


array(['DEPT OF DESIGN & CONSTRUCTION', 'DEPARTMENT OF TRANSPORTATION',
       'OFFICE OF LABOR RELATIONS', 'NYC EMPLOYEES RETIREMENT SYS',
       'FIRE DEPARTMENT', 'DEPT OF ENVIRONMENT PROTECTION',
       'CONSUMER AND WORKER PROTECTION', 'DEPT. OF HOMELESS SERVICES',
       'DEPT OF HEALTH/MENTAL HYGIENE', 'HOUSING PRESERVATION & DVLPMNT',
       "ADMIN FOR CHILDREN'S SVCS", 'HRA/DEPT OF SOCIAL SERVICES',
       'DEPARTMENT OF INVESTIGATION', 'LAW DEPARTMENT',
       'DISTRICT ATTORNEY-MANHATTAN', 'OFF OF PAYROLL ADMINISTRATION',
       'DEPARTMENT OF CITY PLANNING', 'HUMAN RIGHTS COMMISSION',
       'DISTRICT ATTORNEY RICHMOND COU', 'LANDMARKS PRESERVATION COMM',
       'TAXI & LIMOUSINE COMMISSION', 'NYC HOUSING AUTHORITY',
       'BRONX DISTRICT ATTORNEY', 'OFFICE OF THE COMPTROLLER',
       'MUNICIPAL WATER FIN AUTHORITY', 'DEPARTMENT OF FINANCE',
       'DEPARTMENT OF BUILDINGS', 'DEPARTMENT OF CORRECTION',
       'DEPT OF PARKS & RECREATION', 'DEPARTMENT FOR THE AGING',
      

In [ ]:
print(f'Number of unique business titles: {jobs_df['Business Title'].nunique()}')
print(jobs_df['Business Title'].unique())

Number of unique business titles: 1094
['Senior Project Manager' 'Senior Data Analyst' 'Timekeeper' ...
 'Human Resources Generalist' 'Digital Forensic Examiner II'
 'Property Tax Graduate Intern']


In [ ]:
print(f'Number of unique civil titles: {jobs_df['Civil Service Title'].nunique()}')
jobs_df['Civil Service Title'].unique()

Number of unique civil titles: 295


array(['ASSOCIATE PROJECT MANAGER', 'CITY RESEARCH SCIENTIST',
       'PRINCIPAL ADMINISTRATIVE ASSOC', 'COMMUNITY COORDINATOR',
       'ADMINISTRATIVE PROJECT MANAGER', 'COMPUTER SYSTEMS MANAGER',
       'AGENCY ATTORNEY', 'CIVIL ENGINEERING INTERN',
       'CLERICAL ASSOCIATE', 'ENVIRONMENTAL ENGINEERING INTE',
       'ADMINISTRATIVE ENGINEER', 'PROCUREMENT ANALYST', 'EXTERMINATOR',
       'ADMINISTRATIVE BUSINESS PROMOT', 'EXECUTIVE AGENCY COUNSEL',
       'CRIMINALIST', 'COMPUTER SPECIALIST (SOFTWARE)',
       'PUBLIC HEALTH SANITARIAN', 'DIRECTOR OF FIELD OPERATIONS (',
       'HEALTH SERVICES MANAGER NON MA', 'CONFIDENTIAL INVESTIGATOR',
       'ASSISTANT CORPORATION COUNSEL', 'COMMUNITY ASSOCIATE',
       'COLLEGE AIDE (ALL CITY DEPTS)', 'ADMINISTRATIVE PUBLIC HEALTH N',
       'DEPUTY EXECUTIVE DIRECTOR (OPA', 'SUPERVISOR II SOCIAL WORK',
       'CITY PLANNER', 'CONSTRUCTION PROJECT MANAGER',
       'INDUSTRIAL HYGIENIST', 'SPECIAL ASSISTANT TO THE DISTR',
       'ASSISTANT CIV

In [ ]:
def classify_title_group(title):

    title_upper = title.upper()

    groups = {
        'Project Managers': ['PROJECT MANAGER', 'ADM CONSTRUCTION PRJ MGR'],
        'Analysts': ['ANALYST', 'STAFF ANALYST', 'BUSINESS RESEARCH', 'DATA ANALY'],
        'Engineers': ['ENGINEER', 'ENGINEERING', 'INSTRUMENTATION'],
        'IT Specialists': ['IT ', 'COMPUTER', 'SOFTWARE', 'CYBER', 'AUTOMATION'],
        'Legal': ['ATTORNEY', 'COUNSEL', 'LEGAL', 'PARALEGAL', 'EXAMINER', 'CORRECTIONAL STANDARDS'],
        'Healthcare': ['NURSE', 'PHYSICIAN', 'MEDICAL', 'HEALTH', 'DIETITIAN', 'PSYCHOLOGIST',
                      'PHARMACY', 'X-RAY', 'LABORATORY', 'MORTUARY', 'SCIENTIST'],
        'Inspectors & Investigators': ['INSPECTOR', 'INVESTIGATOR', 'AUDITOR', 'CRIMINALIST',
                                       'PROTECTION AGENT', 'SPECIAL OFFICER', 'SAFETY AGENT'],
        'Administrative Support': ['CLERICAL', 'SECRETARY', 'BOOKKEEPER', 'ADMINISTRATIVE',
                                   'ADMIN ', 'ADM ', 'STOREKEEPER', 'RECORDS OFFICER',
                                   'EXECUTIVE ASSISTANT', 'STAFF ASSISTANT', 'MAYORAL OFFICE'],
        'Social Workers': ['CASEWORKER', 'CHILD', 'YOUTH', 'SOCIAL WORK', 'ELIGIBILITY',
                          'PEER COUNSELOR', 'COMMUNITY ASSISTANT', 'COMMUNITY ASSOCIATE',
                          'COMMUNITY COORDINATOR', 'PLAYGROUND'],
        'Managers & Directors': ['DIRECTOR', 'COMMISSIONER', 'EXECUTIVE DIRECTOR', 'DEPUTY',
                                 'SUPERVISOR', 'CHIEF OF', 'CAPTAIN', 'TRANSACTIONS MANAGER'],
        'Planning & Architecture': ['PLANNER', 'ARCHITECT', 'SURVEYOR', 'GEOLOGIST',
                                    'URBAN DESIGNER', 'LANDSCAPE', 'HORTICULTURIST'],
        'Finance & Procurement': ['ACCOUNTANT', 'PROCUREMENT', 'CONTRACT SPECIALIST',
                                  'COLLECTIONS', 'CLAIM SPECIALIST', 'ECONOMIST',
                                  'RETIREMENT BENEFITS', 'BENEFITS'],
        'Trades & Maintenance': ['CARPENTER', 'ELECTRICIAN', 'MACHINIST', 'LABORER',
                                 'MAINTAINER', 'MAINTENANCE WORKER', 'COOK', 'GARDENER',
                                 'CLIMBER', 'CUSTODIAN', 'CARETAKER', 'CITY PARK WORKER',
                                 'HEATING PLANT', 'HIGH PRESSURE PLANT', 'AUTOMOTIVE',
                                 'MOTOR VEHICLE', 'TRAFFIC DEVICE', 'ENERGY CONSERVATION',
                                 'EXTERMINATOR', 'PEST CONTROL', 'WATERSHED'],
        'Transport & Marine': ['MARINER', 'DECKHAND', 'MARINE OILER', 'HOSTLER',
                               'TRANSPORTATION SPECIALIST', 'SLUDGE', 'PORT MARINE'],
        'Housing': ['HOUSING', 'RESIDENT BUILDING', 'CARETAKER (HA)'],
        'Research & Science': ['RESEARCH', 'SCIENTIST', 'GEOLOGIST', 'ECONOMIST',
                               'INDUSTRIAL HYGIENIST'],
        'Communications & Media': ['MEDIA SERVICES', 'BUSINESS PROMOTION', 'PUBLIC RECORDS',
                                   'COMMUNITY RELATIONS'],
        'Emergency & Public Safety': ['EMERGENCY PREPAREDNESS', 'FIRE PROTECTION'],
        'Interns & Aides': ['INTERN', 'AIDE', 'TRAINEE', 'APPRENTICE', 'CITY SEASONAL'],
    }

    for group, keywords in groups.items():
        if any([(kw in title_upper) for kw in keywords]):
            return group

    return 'Other'


jobs_df['Title Group'] = jobs_df['Civil Service Title'].apply(classify_title_group)


print(jobs_df['Title Group'].value_counts())


remaining = jobs_df[jobs_df['Title Group'] == 'Other']['Civil Service Title'].unique()
print(f"\n Other: {len(remaining)}")
print(remaining)

Title Group
Interns & Aides               212
Social Workers                174
Healthcare                    150
Administrative Support        149
Engineers                     112
Legal                          90
Other                          74
Project Managers               72
Inspectors & Investigators     60
Analysts                       54
Trades & Maintenance           50
IT Specialists                 42
Research & Science             39
Finance & Procurement          37
Managers & Directors           34
Planning & Architecture        24
Housing                        13
Transport & Marine              7
Communications & Media          4
Emergency & Public Safety       2
Name: count, dtype: int64

 Other: 18
['SPECIAL ASSISTANT TO THE DISTR' 'ASSISTANT ENVIRONMENTAL ENGINE'
 'EXECUTIVE PROGRAM SPECIALIST' 'CONFIDENTIAL AGENCY INVESTIGAT'
 'RECREATION SPECIALIST (DEPT OF' 'Executive Program Specialist'
 'STOCK WORKER' 'CITY ATTENDANT' 'PRINCIPAL PLANNING CONSULTANT'
 'Specia

The `Civil Service Title` column contained 295 unique standardized job titles.
To make this field more suitable for analysis, titles were grouped into 19 broader
categories based on keyword matching. Each title was assigned to a group by scanning
for relevant keywords, with unmatched titles labeled as `Other`.

A new column `Title Group` was created and used as an additional dimension
for exploring salary patterns and hiring demand across different job functions.

In [ ]:
jobs_df['Residency Requirement'].unique()

array(['New York City Residency is not required for this position',
       'New York City residency is generally required within 90 days of appointment. However, City Employees in certain titles who have worked for the City for 2 continuous years may also be eligible to reside in Nassau, Suffolk, Putnam, Westchester, Rockland, or Orange County. To determine if the residency requirement applies to you, please discuss with the agency representative at the time of interview.',
       'New York City Residency is not required for this position.',
       '•\tCity Residency is not required for this position.',
       'NYC Residency is not required for this title.',
       'New York City residency is not required for this position.',
       'NYCHA has no residency requirements.',
       'City Residency is not required for this position',
       'City Residency is not required for this position.',
       'New York City Residency is not required for this title.',
       'Residency in New York Ci

In [ ]:

def classify_residency(text):

    text_lower = text.lower().strip()

    # Check NOT required first (to avoid false positives)
    not_required_patterns = [
        r'not required',
        r'no residency requirement',
        r'no nyc residency',
        r'no new york city residency',
        r'nycha has no residency',
        r'nycha does not have residency',
        r'the trust has no residency',
        r'exempt from nyc residency',
        r'does not require',
        r'not require residency',
    ]

    for pattern in not_required_patterns:
        if re.search(pattern, text_lower):
            return 'Not Required'

    # Then check IS required
    required_patterns = [
        r'residency (is |)(required|generally required)',
        r'nyc residency is required',
        r'new york (city|state) residency (is |)(required|generally required)',
        r'residency in new york city',
        r'required within 90 days',
        r'residency required',
    ]

    for pattern in required_patterns:
        if re.search(pattern, text_lower):
            return 'Required'

    return 'Not specified'



jobs_df['Residency Category'] = jobs_df['Residency Requirement'].apply(classify_residency)

print(jobs_df['Residency Category'].value_counts())

Residency Category
Required         800
Not Required     596
Not specified      3
Name: count, dtype: int64


markdownThe `Residency Requirement` column contained unstructured free-text descriptions with
over 50 unique variations of the same information. To make this field usable for analysis,
a rule-based classifier was built using regular expressions to parse each value and assign
it to one of three standardized categories: `Required`, `Not Required`, or `Not Specified`.

A new column `Residency Category` was created and used as a categorical variable
for filtering and visualization.

In [ ]:
# Creating a dataset for Tableau Dashboard
jobs_df_tableau = jobs_df[['Job ID', 'Agency', 'Title Group', '# Of Positions', 'Work Schedule Type', 'Career Level', 'Salary Range From', 'Salary Frequency' ,'Salary Range To', 'Posting Date', 'Posting Updated', 'Primary Job Category', 'Residency Category']]

In [ ]:
# Fixing Agency names for visualization purposes

def title_except_small(text):

    small_words = ['of', 'and', 'the', 'a', 'an', 'in', 'on', 'at', 'for', 'to', 'with', '&']
    words = text.split()
    return ' '.join(word.capitalize() if word.lower() not in small_words else word.lower() for word in words)

jobs_df_tableau['Agency'] = jobs_df_tableau['Agency'].apply(title_except_small)

/tmp/ipykernel_8027/2406892020.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  jobs_df_tableau['Agency'] = jobs_df_tableau['Agency'].apply(title_except_small)


In [ ]:
jobs_df_tableau.head()

,Job ID,Agency,Title Group,# Of Positions,Work Schedule Type,Career Level,Salary Range From,Salary Frequency,Salary Range To,Posting Date,Posting Updated,Primary Job Category,Residency Category
873,759255,Dept of Design & Construction,Project Managers,1,F,Experienced (non-manager),86749.0,Annual,99761.0,2026-01-16,2026-03-04,"Engineering, Architecture, & Planning",Not Required
2011,763664,Department of Transportation,Healthcare,1,F,Experienced (non-manager),87743.0,Annual,110120.0,2026-01-17,2026-01-20,"Policy, Research & Analysis",Required
647,766755,Office of Labor Relations,Administrative Support,1,F,Experienced (non-manager),61656.0,Annual,89625.0,2026-01-17,2026-01-17,Administration & Human Resources,Required
989,766758,Office of Labor Relations,Social Workers,1,F,Entry-Level,62868.0,Annual,97593.0,2026-01-17,2026-01-17,Administration & Human Resources,Required
1110,762886,Department of Transportation,Project Managers,1,F,Experienced (non-manager),62407.0,Annual,172854.0,2026-01-17,2026-01-20,"Engineering, Architecture, & Planning",Not Required


# Text Analysis

##Preferred skills

In [ ]:
# Function to clean and standardize text data

def clean_text(text):
  text = str(text).lower()

  # Replace all characters except letters, numbers, and spaces with a space
  # This removes punctuation and special symbols that are usually not useful for word frequency analysis
  text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)

  # Replace multiple spaces with a single space and remove leading/trailing spaces
  text = re.sub(r'\s+', ' ', text).strip()
  return text

jobs_df['pf_cleaned'] = jobs_df['Preferred Skills'].apply(clean_text)

In [ ]:
# Function to extract the most frequent n-grams from a text column

def get_top_ngrams(series, ngram_range=(2,2), min_df=3, n=20):

    # - stop_words='english' removes common English words
    # - ngram_range=(2,2) means only bigrams will be extracted
    #   for example: "project manager", "data analyst"
    # - min_df=3 keeps only n-grams that appear in at least 3 rows

    vectorizer = CountVectorizer(
        stop_words='english',
        ngram_range=ngram_range,
        min_df=min_df
    )
    X = vectorizer.fit_transform(series)

    ngram_counts = pd.DataFrame({
        'ngram': vectorizer.get_feature_names_out(),
        'count': X.toarray().sum(axis=0)
    }).sort_values('count', ascending=False)

    return ngram_counts.head(n)

top_bigrams_skills = get_top_ngrams(jobs_df['pf_cleaned'], ngram_range=(2,2), min_df=3, n=60)


print("Top bigrams in Preferred skills:")
print(top_bigrams_skills)



Top bigrams in Preferred skills:
                            ngram  count
380          communication skills    336
70                   ability work    233
1355                     new york    198
1298             microsoft office    186
1913               skills ability    175
1060         interpersonal skills    147
2319                   word excel    119
2293               verbal written    112
627                driver license    112
1667        proficiency microsoft    108
2385        written communication    107
1630              problem solving    103
1436        organizational skills     99
1398                 office suite     98
2380               writing skills     95
2339           work independently     93
830            experience working     85
2411                    york city     84
1692           project management     83
1309          minimal supervision     79
2395               written verbal     79
1939             skills including     78
2365            working 

In [ ]:
jobs_df['avg_salary'] = (jobs_df['Salary Range From'] + jobs_df['Salary Range To']) / 2

salary_median = jobs_df['avg_salary'].median()

high_salary = jobs_df[jobs_df['avg_salary'] >= salary_median].copy()
low_salary = jobs_df[jobs_df['avg_salary'] < salary_median].copy()

high_salary_skills = get_top_ngrams(high_salary['pf_cleaned'])
low_salary_skills = get_top_ngrams(low_salary['pf_cleaned'])
high_salary_skills


,ngram,count
209,communication skills,166
795,new york,137
1125,skills ability,96
49,ability work,90
757,microsoft office,70
358,driver license,67
1009,project management,64
1338,verbal written,60
1398,written communication,60
613,interpersonal skills,60


In [ ]:
low_salary_skills

,ngram,count
135,communication skills,170
28,ability work,143
497,microsoft office,116
407,interpersonal skills,87
739,skills ability,79
926,word excel,73
953,writing skills,72
544,office suite,70
503,minimal supervision,69
642,proficiency microsoft,67


In [ ]:
skills_df = pd.read_csv('extracted_skills.csv')

## Skills Extraction Using AI

To enable deeper analysis of the skills landscape across NYC job postings,
a separate dataset was created using AI-assisted extraction.

Each row in the `Preferred Skills` column was passed to an LLM (Claude),
which parsed the unstructured text and extracted structured information
into the following categories: `technical_skills`, `tools_platforms`, and `soft_skills`.

The output was returned in JSON format and stored as a new dataset containing
703 rows, which was then used for skills-based analysis.

In [ ]:
skills_df

,Skill Description,Extracted Skills JSON
0,-Ability to understand and analyze architectur...,"{\n ""technical_skills"": [],\n ""tools_platfor..."
1,Preferred Skills / Qualifications 1. Strong ...,"{\n ""technical_skills"": [\n ""program manag..."
2,•\tYou can build and maintain a wide array of ...,"{\n ""technical_skills"": [],\n ""tools_platfor..."
3,"1.\tExperience using ropes, harnesses, belay s...","{\n ""technical_skills"": [\n ""experience us..."
4,Candidates should have knowledge of design pri...,"{\n ""technical_skills"": [],\n ""tools_platfor..."
...,...,...
698,•\tAble to communicate well both verbally and ...,"{\n ""technical_skills"": [],\n ""tools_platfor..."
699,•\tYou can maintain working relationships with...,"{\n ""technical_skills"": [],\n ""tools_platfor..."
700,1.\tAdvanced analytical and organizational ski...,"{\n ""technical_skills"": [],\n ""tools_platfor..."
701,"•\tYou have excellent written, oral, and inter...","{\n ""technical_skills"": [],\n ""tools_platfor..."


In [ ]:
skillsdict = skills_df.set_index('Skill Description')['Extracted Skills JSON'].to_dict()

In [ ]:
import json
from collections import Counter

all_skills = []

# Assuming the second column contains the JSON skills
# skills_df.iloc[:, 1] refers to the second column
for skill_json_str in skills_df.iloc[:, 1]:
    try:
        # Parse the JSON string into a dictionary/list
        skill_data = json.loads(skill_json_str)

        # Handle different potential JSON structures (list of skills or dict with categories)
        if isinstance(skill_data, list):
            all_skills.extend(skill_data)
        elif isinstance(skill_data, dict):
            # If it has categories, flatten all values into one list
            for category_skills in skill_data.values():
                if isinstance(category_skills, list):
                    all_skills.extend(category_skills)
                else:
                    all_skills.append(category_skills)
    except (json.JSONDecodeError, TypeError):
        continue

# Create a DataFrame for the counts
skill_counts = pd.Series(all_skills).value_counts().reset_index()
skill_counts.columns = ['Skill', 'Count']

display(skill_counts.head(40))

,Skill,Count
0,communication,204
1,written communication,140
2,interpersonal skills,113
3,Microsoft Office,110
4,Excel,107
5,writing,107
6,organizational skills,106
7,verbal communication,93
8,attention to detail,89
9,interpersonal,88


In [ ]:
import json

data = []
for index, row in jobs_df.iterrows():
  skilltext = row['Preferred Skills']
  if skilltext not in skillsdict:
    continue
  value = skillsdict[skilltext].removeprefix('```json').removesuffix('```').strip()
  skills = json.loads(value)
  for skill_group, skill_list in skills.items():
    for skill in skill_list:
        data.append({
            'Job ID': row['Job ID'],
            'Skill Group': skill_group,
            'Skill': skill,
            'Salary Range From': row['Salary Range From'],
            'Salary Frequency': row['Salary Frequency'],
            'Salary Range To': row['Salary Range To']
         })
df_data = pd.DataFrame(data)

In [ ]:
# Leveling all types of salary frequency to Annual

factor_map = {
    'Annual': 1,
    'Hourly': 40 * 52,
    'Daily': 260
}

df_data['factor'] = df_data['Salary Frequency'].map(factor_map)

df_data['Annual Salary From'] = df_data['Salary Range From'] * df_data['factor']
df_data['Annual Salary To'] = df_data['Salary Range To'] * df_data['factor']

# Creating Average Annual Salary
df_data['Average Annual Salary'] = (
    df_data['Annual Salary From'] + df_data['Annual Salary To']
) / 2

In [ ]:
## df_grp = df_data.groupby('Skill').agg({'Annual Salary From': 'mean', 'Annual Salary To': 'mean', 'Average Annual Salary': 'mean', 'Job ID': 'count'})
## df_grp[df_grp['Job ID'] > 10].sort_values(by='Average Annual Salary', ascending=False).head(50)

In [ ]:
# Check

df_data[df_data['Salary Frequency'] == 'Hourly']

,Job ID,Skill Group,Skill,Salary Range From,Salary Frequency,Salary Range To,factor,Annual Salary From,Annual Salary To,Average Annual Salary
478,767849,soft_skills,communication,19.14,Hourly,22.47,2080,39811.2,46737.6,43274.4
479,767849,soft_skills,interpersonal skills,19.14,Hourly,22.47,2080,39811.2,46737.6,43274.4
480,767849,soft_skills,problem solving,19.14,Hourly,22.47,2080,39811.2,46737.6,43274.4
481,767849,soft_skills,comprehension,19.14,Hourly,22.47,2080,39811.2,46737.6,43274.4
482,767849,domain_knowledge,trades,19.14,Hourly,22.47,2080,39811.2,46737.6,43274.4
...,...,...,...,...,...,...,...,...,...,...
8816,773251,soft_skills,verbal communication,17.90,Hourly,26.73,2080,37232.0,55598.4,46415.2
8817,773251,soft_skills,interpersonal communication,17.90,Hourly,26.73,2080,37232.0,55598.4,46415.2
8818,773251,soft_skills,customer service,17.90,Hourly,26.73,2080,37232.0,55598.4,46415.2
8819,773251,soft_skills,creativity,17.90,Hourly,26.73,2080,37232.0,55598.4,46415.2


In [ ]:
df_data.groupby('Skill').agg({'Annual Salary From': 'mean', 'Annual Salary To': 'mean', 'Average Annual Salary': 'mean', 'Job ID': 'count'}).sort_values(by='Job ID', ascending=False).head(50)

,Annual Salary From,Annual Salary To,Average Annual Salary,Job ID
Skill,,,,
communication,74830.355373,99194.280695,87012.318034,233
written communication,74295.626299,94809.129150,84552.377725,167
Microsoft Office,62706.860923,81354.180308,72030.520615,130
Excel,66129.121587,82835.462857,74482.292222,126
interpersonal skills,65606.638033,86938.031475,76272.334754,122
writing,57658.268746,72413.286712,65035.777729,118
organizational skills,73095.467583,93539.931722,83317.699652,115
verbal communication,72209.464793,92000.497009,82104.980901,111
attention to detail,61556.204114,75647.622933,68601.913524,105


Before creating the skill mapping, I filtered out rare skills and kept only those appearing in at least 4 job postings to reduce noise and focus on relevant patterns.


In [ ]:
#  df_data.groupby('Skill').agg({'Annual Salary From': 'mean', 'Annual Salary To': 'mean', 'Average Annual Salary': 'mean', 'Job ID': 'count'}).sort_values(by='Job ID', ascending=False)
# skill_list = df_group[df_group['Job ID'] > 3]['Skill'].tolist()
# df_group = df_data.groupby('Skill').agg({'Annual Salary From': 'mean', 'Annual Salary To': 'mean', 'Average Annual Salary': 'mean', 'Job ID': 'count'}).sort_values(by='Job ID', ascending=False)
# df_group[df_group['Job ID'] < 3].index.tolist()

After filtering, I used AI assistance to help group similar skills into broader categories.

In [ ]:
skill_map = {

    # Communication
    'communication': 'Communication',
    'communication skills': 'Communication',
    'verbal communication': 'Communication',
    'oral communication': 'Communication',
    'oral skills': 'Communication',
    'public speaking': 'Communication',
    'presentation': 'Communication',

    # Written Communication
    'written communication': 'Written Communication',
    'writing': 'Written Communication',
    'writing skills': 'Written Communication',
    'written skills': 'Written Communication',

    # Interpersonal & Teamwork
    'interpersonal': 'Interpersonal & Teamwork',
    'interpersonal skills': 'Interpersonal & Teamwork',
    'interpersonal communication': 'Interpersonal & Teamwork',
    'teamwork': 'Interpersonal & Teamwork',
    'collaboration': 'Interpersonal & Teamwork',
    'team player': 'Interpersonal & Teamwork',
    'relationship building': 'Interpersonal & Teamwork',

    # Organizing skills
    'organizational': 'Organizational skills',
    'organizational skills': 'Organizational skills',
    'organization': 'Organizational skills',
    'organized': 'Organizational skills',

    # Time / Task
    'time management': 'Time & Task Management',
    'prioritization': 'Time & Task Management',
    'planning': 'Time & Task Management',
    'multitasking': 'Time & Task Management',
    'multi-tasking': 'Time & Task Management',

    # Detail
    'attention to detail': 'Attention to Detail',
    'detail oriented': 'Attention to Detail',
    'detail-oriented': 'Attention to Detail',

    # Analytical Thinking
    'analytical': 'Analytical Thinking',
    'analytical skills': 'Analytical Thinking',
    'critical thinking': 'Analytical Thinking',
    'judgment': 'Analytical Thinking',

    # Problem Solving
    'problem-solving': 'Problem Solving',
    'problem solving': 'Problem Solving',

    # Research
    'research': 'Research',
    'legal research': 'Research',

    # Leadership
    'leadership': 'Leadership',
    'management': 'Management',
    'project management': 'Management',
    'strategic thinking': 'Strategic Thinking',

    # WORK STYLE
    'ability to work independently': 'Independence',
    'independence': 'Independence',
    'ability to work with minimal supervision': 'Independence',

    'self-motivation': 'Self-Motivation',
    'self motivation': 'Self-Motivation',
    'self-motivated': 'Self-Motivation',
    'initiative': 'Self-Motivation',

    'work ethic': 'Work Ethic',

    # Adaptability
    'adaptability': 'Adaptability',
    'flexibility': 'Adaptability',
    'willingness to learn': 'Adaptability',

    # Professional traits
    'professionalism': 'Professionalism',
    'discretion': 'Professionalism',

    # Customer Service
    'customer service': 'Customer Service',

    # Government
    'government': 'Government / Public Sector',
    'procurement': 'Government / Public Sector',

    # SOFTWARE

    # Microsoft Office (core)
    'microsoft office': 'Microsoft Office',
    'ms office': 'Microsoft Office',
    'microsoft office suite': 'Microsoft Office',
    'microsoft word': 'Microsoft Office',
    'ms word': 'Microsoft Office',
    'word': 'Microsoft Office',
    'excel': 'Microsoft Office',
    'microsoft excel': 'Microsoft Office',
    'microsoft project': 'Presentation / Productivity Tools',

    # Presentation / Productivity Tools
    'powerpoint': 'Presentation / Productivity Tools',
    'microsoft powerpoint': 'Presentation / Productivity Tools',
    'outlook': 'Presentation / Productivity Tools',
    'microsoft outlook': 'Presentation / Productivity Tools',
    'adobe acrobat': 'Presentation / Productivity Tools',
    'visio': 'Presentation / Productivity Tools',

    # DATA / TECH
    'sql': 'SQL',
    'python': 'Programming',
    'r': 'Programming',

    'data analysis': 'Data Analysis',
    'data analytics': 'Data Analysis',
    'data management': 'Data Analysis',

    'tableau': 'BI / Visualization',
    'power bi': 'BI / Visualization',
    'data visualization': 'BI / Visualization',

    'data entry': 'Administrative Data Work',
    'computer skills': 'Administrative Data Work',
    'fms': 'Administrative Data Work',

    # ENGINEERING

    # Core engineering
    'civil engineering': 'Engineering',
    'mechanical engineering': 'Engineering',
    'environmental engineering': 'Engineering',
    'engineering practices': 'Engineering',

    # Infrastructure
    'infrastructure': 'Infrastructure & Construction',
    'nyc infrastructure system': 'Infrastructure & Construction',
    'construction': 'Infrastructure & Construction',

    # GIS
    'gis': 'GIS',
    'arcgis': 'GIS',

    # CAD
    'autocad': 'CAD',

    # Scheduling
    'primavera p6': 'Project Controls',

    # Standards
    'aashto standards': 'Engineering Standards',
    'nys dot specifications': 'Engineering Standards',
    'nyc dot specifications': 'Engineering Standards',
    'nyc dep specifications': 'Engineering Standards',

    # Sustainability
    'envision sustainability professional': 'Sustainability',

    # FIELD
    "valid driver's license": "Driver's License"
}

In [ ]:
skill_map_update = {

    # ---------------- COMMUNICATION ----------------
    'strong oral communication': 'Communication',
    'strong oral communication skills': 'Communication',
    'strong verbal communication': 'Communication',
    'verbal': 'Communication',
    'verbal skills': 'Communication',
    'effective communication': 'Communication',
    'strong communication': 'Communication',
    'communications': 'Communication',
    'listening': 'Communication',
    'oral presentation': 'Communication',
    'presentation skills': 'Communication',
    'presentation development': 'Communication',

    'strong writing': 'Written Communication',
    'strong writing skills': 'Written Communication',
    'strong written communication': 'Written Communication',
    'written': 'Written Communication',
    'report writing': 'Written Communication',
    'technical writing': 'Written Communication',
    'legal writing': 'Written Communication',
    'editing': 'Written Communication',

    # ---------------- INTERPERSONAL / TEAMWORK ----------------
    'ability to work as part of a team': 'Interpersonal & Teamwork',
    'ability to work collaboratively': 'Interpersonal & Teamwork',
    'ability to work in a team': 'Interpersonal & Teamwork',
    'ability to work with others': 'Interpersonal & Teamwork',
    'building relationships': 'Interpersonal & Teamwork',
    'collaborative': 'Interpersonal & Teamwork',
    'collaborative skills': 'Interpersonal & Teamwork',
    'collaborative work': 'Interpersonal & Teamwork',
    'cross-functional collaboration': 'Interpersonal & Teamwork',
    'relationship management': 'Interpersonal & Teamwork',
    'stakeholder engagement': 'Interpersonal & Teamwork',
    'stakeholder management': 'Interpersonal & Teamwork',
    'team-oriented': 'Interpersonal & Teamwork',

    # ---------------- ORGANIZATION / EXECUTION ----------------
    'well organized': 'Organizational skills',
    'organizational abilities': 'Organizational skills',
    'ability to prioritize': 'Time & Task Management',
    'deadline management': 'Time & Task Management',
    'meeting deadlines': 'Time & Task Management',
    'ability to meet deadlines': 'Time & Task Management',
    'schedule management': 'Time & Task Management',
    'scheduling': 'Time & Task Management',
    'task management': 'Time & Task Management',
    'coordination': 'Time & Task Management',
    'project coordination': 'Time & Task Management',

    'detail orientation': 'Attention to Detail',
    'attentive to detail': 'Attention to Detail',
    'accuracy': 'Attention to Detail',

    # ---------------- ANALYTICAL / RESEARCH ----------------
    'analytical problem solving': 'Problem Solving',
    'problem-solving skills': 'Problem Solving',
    'database research': 'Research',
    'research skills': 'Research',
    'quantitative': 'Analytical Thinking',
    'quantitative skills': 'Analytical Thinking',
    'quantitative research': 'Research',
    'logical thinking': 'Analytical Thinking',
    'decision making': 'Analytical Thinking',
    'decision-making': 'Analytical Thinking',
    'good judgment': 'Analytical Thinking',
    'sound judgement': 'Analytical Thinking',
    'sound judgment': 'Analytical Thinking',
    'independent judgment': 'Analytical Thinking',
    'statistical analysis': 'Data Analysis',
    'regression analysis': 'Data Analysis',
    'statistics': 'Data Analysis',
    'data science': 'Data Analysis',
    'data acquisition': 'Data Analysis',
    'data governance': 'Data Analysis',
    'data architecture': 'Data Analysis',
    'querying large datasets': 'Data Analysis',
    'databases': 'Data Analysis',
    'relational databases': 'Data Analysis',

    # ---------------- LEADERSHIP / MANAGEMENT ----------------
    'superior leadership': 'Leadership',
    'leadership skills': 'Leadership',
    'leadership development': 'Leadership',
    'developing leadership': 'Leadership',
    'identifying leadership': 'Leadership',
    'supervision': 'Management',
    'supervisory skills': 'Management',
    'supervisory experience': 'Management',
    'management experience': 'Management',
    'managerial experience': 'Management',
    'team management': 'Management',
    'program management': 'Management',
    'budget management': 'Management',
    'financial management': 'Management',
    'facilities management': 'Management',
    'fleet management': 'Management',
    'partnership management': 'Management',
    'event management': 'Management',
    'strategic planning': 'Strategic Thinking',
    'implementation': 'Strategic Thinking',
    'process improvement': 'Strategic Thinking',
    'program development': 'Strategic Thinking',
    'program evaluation': 'Strategic Thinking',
    'business development': 'Strategic Thinking',
    'curriculum development': 'Strategic Thinking',

    'resourcefulness': 'Adaptability',
    'solution orientation': 'Problem Solving',
    'solution oriented': 'Problem Solving',

    # ---------------- WORK STYLE ----------------
    'self-directed': 'Independence',
    'independent work': 'Independence',
    'independently': 'Independence',
    'ability to work well with minimal supervision': 'Independence',

    'self-starter': 'Self-Motivation',
    'self-management': 'Self-Motivation',

    'strong work ethic': 'Work Ethic',
    'accountability': 'Work Ethic',
    'reliability': 'Work Ethic',
    'integrity': 'Work Ethic',
    'strong ethics': 'Work Ethic',

    'ability to work in a fast-paced environment': 'Adaptability',
    'ability to work well in a fast-paced deadline driven environment': 'Adaptability',
    'ability to work under pressure': 'Adaptability',
    'work under pressure': 'Adaptability',
    'staying calm under pressure': 'Adaptability',
    'calm under pressure': 'Adaptability',

    'professional demeanor': 'Professionalism',
    'ability to handle sensitive information': 'Professionalism',
    'confidentiality': 'Professionalism',
    'diplomacy': 'Professionalism',
    'empathy': 'Professionalism',
    'influence': 'Professionalism',
    'negotiation': 'Professionalism',
    'conflict management': 'Professionalism',
    'conflict resolution': 'Professionalism',

    # ---------------- CUSTOMER / SERVICE / COMMUNITY ----------------
    'customer service skills': 'Customer Service',
    'client services': 'Customer Service',
    'community outreach': 'Customer Service',
    'community relations': 'Customer Service',
    'community organizing': 'Customer Service',

    # ---------------- GOVERNMENT / PUBLIC SECTOR ----------------
    'agency operations': 'Government / Public Sector',
    'state agency operations': 'Government / Public Sector',
    'government operations': 'Government / Public Sector',
    'government experience': 'Government / Public Sector',
    'city government': 'Government / Public Sector',
    'new york city government': 'Government / Public Sector',
    'new york city agency operations': 'Government / Public Sector',
    'city legislative process': 'Government / Public Sector',
    'nyc budget process': 'Government / Public Sector',
    'nyc capital budget process': 'Government / Public Sector',
    'nyc procurement rules': 'Government / Public Sector',
    'nyc procurement policy board rules': 'Government / Public Sector',
    "new york city's procurement policy board rules": 'Government / Public Sector',
    'public sector': 'Government / Public Sector',
    'public service': 'Government / Public Sector',
    'public administration': 'Government / Public Sector',
    'city planning': 'Government / Public Sector',
    'housing policy': 'Government / Public Sector',
    'public housing': 'Government / Public Sector',
    'public policy': 'Government / Public Sector',
    'public policy analysis': 'Government / Public Sector',
    'administration': 'Government / Public Sector',

    # ---------------- MICROSOFT / PRODUCTIVITY ----------------
    'ms office 365': 'Microsoft Office',
    'microsoft office 365': 'Microsoft Office',
    'office 365': 'Microsoft Office',
    'microsoft suite': 'Microsoft Office',
    'microsoft programs': 'Microsoft Office',
    'ms excel': 'Microsoft Office',

    'ms outlook': 'Presentation / Productivity Tools',
    'ms powerpoint': 'Presentation / Productivity Tools',
    'microsoft teams': 'Presentation / Productivity Tools',
    'ms teams': 'Presentation / Productivity Tools',
    'teams': 'Presentation / Productivity Tools',
    'sharepoint': 'Presentation / Productivity Tools',
    'onedrive': 'Presentation / Productivity Tools',
    'smartsheet': 'Presentation / Productivity Tools',
    'canva': 'Presentation / Productivity Tools',
    'ms project': 'Presentation / Productivity Tools',
    'project management tools': 'Presentation / Productivity Tools',
    'project': 'Presentation / Productivity Tools',

    # ---------------- ADOBE / DESIGN ----------------
    'adobe': 'Design / Creative Tools',
    'adobe creative suite': 'Design / Creative Tools',
    'adobe photoshop': 'Design / Creative Tools',
    'adobe pro': 'Design / Creative Tools',
    'illustrator': 'Design / Creative Tools',
    'graphic design': 'Design / Creative Tools',
    'design principles': 'Design / Creative Tools',
    'design': 'Design / Creative Tools',
    'creating digital libraries': 'Design / Creative Tools',
    'creativity': 'Design / Creative Tools',

    # ---------------- PROGRAMMING / SOFTWARE DEV ----------------
    '.net': 'Programming',
    'api': 'Programming',
    'angular': 'Programming',
    'c#': 'Programming',
    'css': 'Programming',
    'html': 'Programming',
    'java': 'Programming',
    'javascript': 'Programming',
    'software development': 'Programming',
    'devops': 'Programming',
    'git': 'Programming',

    # ---------------- DATA / BI / DATABASES ----------------
    'powerbi': 'BI / Visualization',
    'pandas': 'Data Analysis',
    'postgresql': 'SQL',
    'machine learning': 'Programming',
    'data science': 'Data Analysis',

    # ---------------- CLOUD / SECURITY / IT ----------------
    'aws': 'Cloud / Security / IT',
    'information security': 'Cloud / Security / IT',
    'cybersecurity': 'Cloud / Security / IT',
    'hipaa': 'Cloud / Security / IT',
    'network+': 'Cloud / Security / IT',
    'itil': 'Cloud / Security / IT',
    'microsoft windows': 'Cloud / Security / IT',
    'windows': 'Cloud / Security / IT',
    'computer forensics': 'Cloud / Security / IT',
    'technology': 'Cloud / Security / IT',
    'computer literacy': 'Cloud / Security / IT',
    'familiarity with computers/software': 'Cloud / Security / IT',
    'troubleshooting': 'Cloud / Security / IT',

    # ---------------- LEGAL / COMPLIANCE ----------------
    'law': 'Legal / Compliance',
    'legal': 'Legal / Compliance',
    'legal issues handling': 'Legal / Compliance',
    'legal rules and regulations': 'Legal / Compliance',
    'litigation': 'Legal / Compliance',
    'litigation experience': 'Legal / Compliance',
    'litigation terminology': 'Legal / Compliance',
    'trial experience': 'Legal / Compliance',
    'eDiscovery': 'Legal / Compliance',
    'juris doctorate': 'Legal / Compliance',
    'nys bar': 'Legal / Compliance',
    'new york state penal law': 'Legal / Compliance',
    'americans with disabilities act': 'Legal / Compliance',
    'compliance': 'Legal / Compliance',
    'regulatory compliance': 'Legal / Compliance',
    'environmental regulations': 'Legal / Compliance',
    'u.s. department of housing and urban development regulations': 'Legal / Compliance',
    'osha': 'Legal / Compliance',
    'administrative law': 'Legal / Compliance',

    # ---------------- FINANCE / OPERATIONS ----------------
    'accounting': 'Finance / Operations',
    'budget': 'Finance / Operations',
    'budgeting': 'Finance / Operations',
    'finance': 'Finance / Operations',
    'financial management system': 'Finance / Operations',
    'nyc financial management system': 'Finance / Operations',
    'fms2': 'Finance / Operations',
    'fms3': 'Finance / Operations',
    'otps tracking': 'Finance / Operations',
    'operations': 'Finance / Operations',
    'technical operations': 'Finance / Operations',
    'records': 'Finance / Operations',
    'document management': 'Finance / Operations',
    'contracts': 'Finance / Operations',
    'contracting': 'Finance / Operations',
    'contract negotiation': 'Finance / Operations',

    # ---------------- ENGINEERING / CONSTRUCTION ----------------
    'engineering': 'Engineering',
    'electrical engineering': 'Engineering',
    'mechanical engineering': 'Engineering',
    'environmental engineering': 'Engineering',
    'chemical engineering': 'Engineering',
    'engineering concepts': 'Engineering',
    'engineering drawings': 'Engineering',
    'engineering plans': 'Engineering',
    'construction': 'Infrastructure & Construction',
    'construction management': 'Infrastructure & Construction',
    'design and construction project management': 'Infrastructure & Construction',
    'infrastructure construction': 'Infrastructure & Construction',
    'infrastructure projects': 'Infrastructure & Construction',
    'infrastructure work': 'Infrastructure & Construction',
    'stormwater management': 'Infrastructure & Construction',
    'wastewater treatment': 'Infrastructure & Construction',
    'wastewater industry': 'Infrastructure & Construction',
    'traffic engineering': 'Infrastructure & Construction',
    'transportation': 'Infrastructure & Construction',
    'transportation planning': 'Infrastructure & Construction',
    'urban design': 'Infrastructure & Construction',
    'architecture': 'Infrastructure & Construction',
    'architectural': 'Infrastructure & Construction',
    'revit': 'CAD',
    'autodesk autocad': 'CAD',
    'cadd': 'CAD',
    'civil 3d': 'CAD',
    'bluebeam revu': 'CAD',
    'aashto': 'Engineering Standards',
    'mutcd': 'Engineering Standards',
    'ceqr technical manual': 'Engineering Standards',
    'zoning resolution': 'Engineering Standards',
    'new york city zoning resolution': 'Engineering Standards',
    'nyc building code': 'Engineering Standards',
    'nyc construction codes': 'Engineering Standards',

    # ---------------- GIS / PLANNING / ENVIRONMENT ----------------
    'arcgis online': 'GIS',
    'city planning': 'GIS',
    'land surveying': 'GIS',
    'survey equipment': 'GIS',
    'environmental science': 'Environmental',
    'biology': 'Environmental',
    'chemistry': 'Environmental',
    'ecology': 'Environmental',
    'science': 'Environmental',
    'public health': 'Environmental',
    'water safety instructor': 'Environmental',

    # ---------------- DOMAIN KNOWLEDGE ----------------
    'affordable housing': 'Domain Knowledge',
    'human resources': 'Domain Knowledge',
    'human services': 'Domain Knowledge',
    'social services': 'Domain Knowledge',
    'social work': 'Domain Knowledge',
    'mental health': 'Domain Knowledge',
    'criminal justice system': 'Domain Knowledge',
    'criminal justice policy': 'Domain Knowledge',
    'law enforcement': 'Domain Knowledge',
    'economic development': 'Domain Knowledge',
    'business administration': 'Domain Knowledge',
    'business': 'Domain Knowledge',
    'community development': 'Domain Knowledge',
    'real estate': 'Domain Knowledge',
    'workforce development': 'Domain Knowledge',
    'sanitation': 'Domain Knowledge',
    'sports': 'Domain Knowledge',
    'soccer': 'Domain Knowledge',

    # ---------------- CERTIFICATIONS / LICENSES ----------------
    'driver license': "Driver's License",
    "driver's license": "Driver's License",
    'motor vehicle driver’s license': "Driver's License",
    "motor vehicle driver's license": "Driver's License",
    'valid new york state driver license': "Driver's License",
    "nys driver's license": "Driver's License",
    'nys driver’s license': "Driver's License",
    'nys pe license': "Driver's License",
    'pmp': 'Field Licences',
    'leed': 'Field Licences',
    'env sp': 'Field Licences',

    # ---------------- LANGUAGES ----------------
    'spanish': 'Languages',
    'bilingual': 'Languages',
    'bilingual abilities': 'Languages',
    'multilingual': 'Languages',
    'fluency in a language in addition to english': 'Languages',
    'fluency in a language in addition to english': 'Languages',
    'foreign language skills': 'Languages',
    'language skills': 'Languages',

    # ---------------- MISC / TO REVIEW ----------------
    'new york city': 'Government / Public Sector',
    'state agency operations': 'Government / Public Sector',
    'm/wbe': 'Government / Public Sector',
    'nycway': 'Government / Public Sector',
    'seams': 'Government / Public Sector',
    'hhs accelerator': 'Government / Public Sector',
    'citytime': 'Government / Public Sector',
    'fhems': 'Government / Public Sector',
    'chrms': 'Government / Public Sector',
    'prise': 'Government / Public Sector',
    'cares': 'Government / Public Sector',
    'e-gordian': 'Government / Public Sector',
    'pms': 'Government / Public Sector',
    'pos': 'Government / Public Sector'
}

In [ ]:
mapping_groups = {
    'Communication': [
        'strong communication skills', 'clear communication', 'verbal communication',
        'explanation of complex topics', 'ability to communicate concisely and effectively',
        'ability to communicate effectively', 'ability to communicate results',
        'ability to explain complex topics simply',
        'ability to synthesize and explain complex information',
        'ability to articulate the law'
    ],

    'Written Communication': [
        'written communication', 'written communication skills', 'written expression',
        'strong written skills', 'strong writing skills',
        'ability to draft quick responses', 'ability to prepare clear reports',
        'reporting'
    ],

    'Interpersonal & Teamwork': [
        'teamwork', 'team skills', 'effective working relationships', 'people skills',
        'ability to work cooperatively', 'ability to work with cross-functional teams',
        'ability to work with diverse groups of people', 'ability to work within teams',
        'consensus building', 'building relationships', 'group work', 'team building',
        'respectful', 'approachability', 'outgoing'
    ],

    'Organizational skills': [
        'highly organized', 'organizational'
    ],

    'Time & Task Management': [
        'handling multiple assignments', 'ability to manage multiple priorities',
        'ability to manage multiple tasks', 'ability to multi-task',
        'ability to multitask', 'ability to balance multiple assignments',
        'ability to handle multiple projects',
        'ability to handle multiple priorities and projects simultaneously',
        'ability to prioritize work', 'ability to prioritize projects',
        'ability to manage deadline-driven projects',
        'ability to manage multiple assignments', 'ability to manage multiple projects',
        'ability to manage workload', 'timeliness', 'follow-up',
        'project follow through', 'project completion'
    ],

    'Attention to Detail': [
        'detail-orientation'
    ],

    'Analytical Thinking': [
        'analysis', 'analytic', 'analytic thinking', 'analytical abilities',
        'analytical ability', 'effective judgment and decision making', 'evaluation'
    ],

    'Problem Solving': [
        'problem analysis'
    ],

    'Research': [
        'research capabilities', 'research experience', 'research literature', 'qualitative',
        'qualitative research'
    ],

    'Data Analysis': [
        'quantitative analysis', 'data retrieval', 'data processing', 'data manipulation',
        'data management systems', 'analytics', 'etl', 'database', 'database development',
        'database manipulation', 'database queries', 'pivot tables', 'vlookup', 'sumifs', 'advanced formatting', 'advanced charts'
    ],

    'BI / Visualization': [
        'data visualization', 'powerbi', 'microsoft powerbi', 'power query',
        'data storytelling'
    ],

    'Leadership': [
        'supervisory leadership'
    ],

    'Management': [
        'people management', 'executive staff management', 'personnel management',
        'management skills', 'managerial', 'supervisory', 'supervisory experience',
        'project management', 'multi-project management', 'vendor management',
        'contract management', 'contract administration'
    ],

    'Strategic Thinking': [
        'scope development', 'change management', 'operational efficiency',
        'workflow automation', 'workflow enhancements', 'policy development',
        'strategic planning'
    ],

    'Adaptability': [
        'ability to adapt', 'ability to learn quickly', 'ability to learn',
        'ability to grasp new concepts', 'willingness to learn',
        'ability to navigate ambiguity', 'ability to respond well to shifting priorities',
    ],

    'Self-Motivation': [
        'proactive approach', 'self-motivation'
    ],

    'Work Ethic': [
        'enthusiastic work ethic', 'ethics', 'responsibility', 'commitment'
    ],

    'Professionalism': [
        'maturity', 'assertive', 'assertiveness'
    ],

    'Microsoft Office': [
        'ms office suite', 'microsoft 365', 'office tools', 'word processing'
    ],

    'Presentation / Productivity Tools': [
        'microsoft visio', 'visio', 'one drive', 'zoom', 'zoom phone', 'webex',
        'microsoft publisher', 'power point', 'prezi', 'smartsheets'
    ],

    'Design / Creative Tools': [
        'photoshop', 'adobe indesign', 'adobe premiere pro', 'adobe professional',
        'after effects', 'indesign', 'video editing', 'content creation',
        'ui/ux design', 'ux design', 'ux/ui design', 'user interface design',
        'blender', '3-d software', '3d rendering', 'c4d'
    ],

    'Programming': [
        'natural language processing', 'ml algorithms', 'pytorch', 'tensorflow',
        'scikit-learn', 'scikit learn', 'huggingface', 'numpy', 'matplotlib',
        'statsmodels', 'sas', 'stata', 'c++', 'php', 'typescript', 'react',
        'react js', 'node js', 'rest apis', 'restful api development',
        'restful apis', 'web apis', 'web services', 'openapi', 'swagger',
        'ci/cd', 'version control', 'scripting', 'software development',
        'software development lifecycle', 'sdlc', 'single page applications',
        'serverless application patterns', 'serverless applications', 'vs code',
        'visual studio', 'xaml', 'xml', 'xsl'
    ],

    'SQL': [
        'sql server', 'ms sql server', 'oracle sql', 'sqlite', 'mysql',
        'sql server studio', 'ssis'
    ],

    'Cloud / Security / IT': [
        'google cloud', 'cloud', 'cloud infrastructure', 'cloud data platforms',
        'firewalls', 'it infrastructure', 'cyber security', 'active directory',
        'virtualization', 'vmware', 'macos', 'ms operating systems', 'windows 10',
        'windows 11', 'windows pcs', 'network routing', 'network switching',
        'network analysis', 'network', 'siem', 'pci-dss', 'rbac', 'pam',
        'service level agreements', 'servicenow'
    ],

    'Government / Public Sector': [
        'citywide procurement systems', 'citywide time and leave regulations',
        'procurement', 'procurement analyst', 'procurement policy board rules',
        'procurement and sourcing solution portal',
        'procurement and sourcing solution portal (passport)', 'city agencies',
        'nyc agencies', 'municipal government', 'agency structure', 'citywide rules',
        'office of citywide procurement (ocp)', 'the city record online',
        'public administration', 'public policy', 'urban planning', 'land use',
        'new york city regulations', 'new york city human rights laws',
        'new york city zoning', 'new york state regulations',
        'state legislative process', 'state procurement laws',
        'working with public officials'
    ],

    'Legal / Compliance': [
        'title vii of the civil rights law of 1964', 'federal and state discovery rules',
        'federal laws', 'federal regulations', 'federal standards', 'regulations',
        'rules & regulations', 'rules of the city of new york',
        'laws and regulations', 'labor laws', 'criminal law',
        'criminal procedure law', 'new york penal law', 'new york state laws',
        'new york state human rights laws', 'new york state bar membership',
        'paralegal certification', 'legal experience', 'legal knowledge',
        'public sector legal work'
    ],

    'Finance / Operations': [
        'fiscal operations', 'financial analysis', 'payroll', 'payroll information',
        'payroll management system', 'pension fund', 'quickbooks', 'revenue',
        'defined contribution', 'asset management', 'inventory management',
        'warehouse experience', 'operations management'
    ],

    'Engineering': [
        'plumbing', 'structural engineering', 'landscape architecture',
        'geotechnical', 'bridge design'
    ],

    'Infrastructure & Construction': [
        'bridge inspections', 'bridge maintenance', 'bridge rehabilitation',
        'capital construction', 'capital projects', 'construction project management',
        'construction processes', 'building construction', 'high-rise construction',
        'building design', 'public building systems', 'construction administration',
        'construction documents', 'design documents', 'field inspection',
        'inspections', 'quantity takeoffs', 'cost estimating',
        'stormwater management', 'wastewater treatment'
    ],

    'Project Controls': [
        'project controls', 'oracle primavera p6'
    ],

    'CAD': [
        'blueprint reading', 'computer-aided drafting', 'civil3d'
    ],

    'GIS': [
        'survey reading', 'map reading', 'carto', 'geospatial analysis',
        'geospatial analytics', 'geopandas'
    ],

    'Engineering Standards': [
        'engineering reports', 'engineering specifications', 'city codes',
        'safety regulations', 'construction safety', 'osha regulations',
        'osha standards', 'leed requirements', 'asme', 'astm',
        'aashto green book', 'swmm', 'aerscreen', 'aermod', 'awwa',
        'nyc zoning resolution'
    ],

    'Environmental': [
        'environment', 'environmental', 'environmental policy',
        'environmental studies', 'environmental health and safety regulations',
        'environmental health and safety', 'green infrastructure',
        'renewable energy systems'
    ],

    'Domain Knowledge': [
        'health policy', 'health care delivery', 'social work', 'public housing',
        'rental subsidy programs', 'section 8', 'cash assistance processes',
        'cash assistance process'
    ],

    'Languages': [
        'korean', 'french', 'russian', 'urdu', 'bengali', 'polish',
        'fluency in french', 'fluency in mandarin chinese', 'fluency in bengali',
        'fluency in spanish', 'spanish proficiency',
        'fluency in a language other than english', 'foreign languages',
        'bi-lingual', 'bi-lingual (spanish)', 'bi-lingual proficiency',
        'bilingual (spanish)'
    ],

    'Education': [
        "master's degree", 'masters degree', 'master’s degree',
        'master’s degree in business', 'master’s degree in public administration',
        'master’s degree in public policy', 'graduate degree',
        'undergraduate degree', 'undergraduate degree in business administration',
        'undergraduate degree in human resources',
        'undergraduate degree in operations management',
        'bachelor degree', "bachelor's degree", 'baccalaureate degree',
        'advanced degree', 'related field'
    ],

    'Field Licences': [
        'project management professional (pmp)', 'project management professional',
        'professional engineer', 'pe license', 'merchant mariner credential',
        'cpr/aed', 'qualified elevator inspector (qei)',
        'qualified retaining wall inspector', 'registered consulting arborist',
        'us epa lead inspector', 'us epa lead risk assessor',
        'us epa rrp certification', 'technician’s certificate', 'notary public',
        'personal vehicle'
    ]
}

In [ ]:
skill_map_update_2 = {
    skill: category
    for category, skills in mapping_groups.items()
    for skill in skills
}

In [ ]:
skill_map.update(skill_map_update)
skill_map.update(skill_map_update_2)

In [ ]:
df_data['Normalized Skill'] = (
    df_data['Skill']
    .str.strip()
    .str.lower()
    .str.replace(r'\s+', ' ', regex=True)
)

df_data['Skill Category'] = df_data['Normalized Skill'].map(skill_map)

In [ ]:
df_data = df_data.drop_duplicates(subset=['Job ID', 'Skill Category'])

In [ ]:
df_data['Skill Category'].unique()

array(['Organizational skills', 'Communication',
       'Infrastructure & Construction', 'Time & Task Management', nan,
       'Programming', 'SQL', 'BI / Visualization',
       'Interpersonal & Teamwork', 'Analytical Thinking', 'Research',
       'Written Communication', 'Adaptability',
       'Government / Public Sector', 'Microsoft Office',
       'Presentation / Productivity Tools', 'Professionalism',
       'Work Ethic', 'Cloud / Security / IT', 'Legal / Compliance',
       'Engineering Standards', 'Data Analysis', 'Problem Solving',
       'Design / Creative Tools', 'Independence', 'GIS', 'Engineering',
       'Attention to Detail', "Driver's License", 'Management',
       'Leadership', 'Finance / Operations', 'Environmental',
       'Self-Motivation', 'Strategic Thinking', 'CAD', 'Languages',
       'Administrative Data Work', 'Customer Service', 'Project Controls',
       'Field Licences', 'Domain Knowledge', 'Education',
       'Sustainability'], dtype=object)

When extracting skills from job postings, multiple skills listed in a single job can often map to the same category. For example:

Word
Excel
Microsoft Office

All belong to the “Microsoft Office” category.

If we count each of these separately, it artificially inflates the importance of that category for a single job.

Solution - to prevent this, we ensure that each skill category is counted only once per job posting.

In [ ]:
category_to_group = {
    'Adaptability': 'Soft Skills',
    'Analytical Thinking': 'Soft Skills',
    'Attention to Detail': 'Soft Skills',
    'Communication': 'Soft Skills',
    'Customer Service': 'Soft Skills',
    'Independence': 'Soft Skills',
    'Interpersonal & Teamwork': 'Soft Skills',
    'Leadership': 'Soft Skills',
    'Management': 'Soft Skills',
    'Organizational skills': 'Soft Skills',
    'Problem Solving': 'Soft Skills',
    'Professionalism': 'Soft Skills',
    'Research': 'Soft Skills',
    'Self-Motivation': 'Soft Skills',
    'Strategic Thinking': 'Soft Skills',
    'Time & Task Management': 'Soft Skills',
    'Work Ethic': 'Soft Skills',
    'Written Communication': 'Soft Skills',

    'Administrative Data Work': 'Technical Skills',
    'BI / Visualization': 'Technical Skills',
    'CAD': 'Technical Skills',
    'Cloud / Security / IT': 'Technical Skills',
    'Data Analysis': 'Technical Skills',
    'Design / Creative Tools': 'Technical Skills',
    'GIS': 'Technical Skills',
    'Microsoft Office': 'Technical Skills',
    'Presentation / Productivity Tools': 'Technical Skills',
    'Programming': 'Technical Skills',
    'SQL': 'Technical Skills',

    'Domain Knowledge': 'Domain Knowledge',
    'Finance / Operations': 'Domain Knowledge',
    'Government / Public Sector': 'Domain Knowledge',
    'Legal / Compliance': 'Domain Knowledge',

    'Engineering': 'Engineering & Construction',
    'Engineering Standards': 'Engineering & Construction',
    'Environmental': 'Engineering & Construction',
    'Infrastructure & Construction': 'Engineering & Construction',
    'Project Controls': 'Engineering & Construction',
    'Sustainability': 'Engineering & Construction',

    "Driver's license": 'Other',
    'Education': 'Other',
    'Field licences': 'Other',
    'Languages': 'Other'
}

In [ ]:
df_data['Skill Group'] = df_data['Skill Category'].map(category_to_group).fillna('Other')

In [ ]:
df_group = df_data.groupby('Skill Category').agg({'Annual Salary From': 'mean', 'Annual Salary To': 'mean', 'Average Annual Salary': 'mean', 'Job ID': 'count'})
df_group.sort_values(by='Job ID')

,Annual Salary From,Annual Salary To,Average Annual Salary,Job ID
Skill Category,,,,
Sustainability,67114.000000,85360.636364,76237.318182,11
Project Controls,69190.391304,92103.478261,80646.934783,23
Education,79980.296000,114882.648000,97431.472000,25
Field Licences,83610.280000,111535.280000,97572.780000,36
SQL,81139.177778,100476.250000,90807.713889,36
Environmental,48079.687805,55298.409756,51689.048780,41
GIS,70558.038095,95812.742857,83185.390476,42
Strategic Thinking,87210.135814,113145.575814,100177.855814,43
Work Ethic,79000.218605,97871.879070,88436.048837,43


In [ ]:
df_skills_tableau = df_data[['Job ID', 'Skill Group','Skill' ,'Annual Salary From', 'Annual Salary To', 'Average Annual Salary', 'Skill Category']]
df_skills_tableau = df_skills_tableau.dropna(subset=['Skill Category'])
df_skills_tableau

,Job ID,Skill Group,Skill,Annual Salary From,Annual Salary To,Average Annual Salary,Skill Category
0,759255,Soft Skills,organizational,86749.0,99761.0,93255.0,Organizational skills
1,759255,Soft Skills,communication,86749.0,99761.0,93255.0,Communication
2,759255,Engineering & Construction,design and construction project management,86749.0,99761.0,93255.0,Infrastructure & Construction
3,759255,Soft Skills,schedule management,86749.0,99761.0,93255.0,Time & Task Management
5,763664,Technical Skills,Python,87743.0,110120.0,98931.5,Programming
...,...,...,...,...,...,...,...
8835,773191,Soft Skills,written communication,42288.0,48631.0,45459.5,Written Communication
8836,773191,Soft Skills,verbal communication,42288.0,48631.0,45459.5,Communication
8837,773191,Soft Skills,detail oriented,42288.0,48631.0,45459.5,Attention to Detail
8838,773191,Soft Skills,multi-tasking,42288.0,48631.0,45459.5,Time & Task Management


In [ ]:
df_skills_tableau.groupby('Skill Group')['Job ID'].count()

,Job ID
Skill Group,
Domain Knowledge,360
Engineering & Construction,318
Other,174
Soft Skills,3068
Technical Skills,1043


In [ ]:
df_skills_tableau.to_csv('df_skills_tableau.csv')
from google.colab import files
files.download('df_skills_tableau.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>